# 11.10 学习笔记
## 1 exercise
### 1.1 linear-regression-scratch
1.1.1如果我们将权重初始化为0，会发生什么，算法依然有效吗？

answer:

这会导致所有样本的预测值都是0
但是在标签和特征不为零的地方平方损失函数的梯度不等于零
所以最后权重可以正常更新

1.1.2假设试图为电压和电流的关系建立一个模型。自动微分可以用来学习模型的参数吗？

answer：

可以。
只要我们建立了一个关于未知参数的可微模型，就可以利用自动微分计算损失函数的梯度，再通过优化算法学习参数
再这个例子中学到的就是电阻R

1.1.3 能基于普朗克定律使用光谱能量密度来确定物体的温度吗？

$$u_{\lambda}(T)=\frac{8\pi hc}{\lambda^{5}}\frac{1}{\exp{\frac{hc}{\lambda k_{B}T}}-1}$$

answer:


我们希望通过光谱数据反推出$T$
假设我们已经通过实验测量到了$$\{(\lambda_{i},u_{i})\}_{i=1}^{n}$$
我们的理论模型是
$$u_{\lambda}(T)=\frac{8\pi hc}{\lambda^{5}}\frac{1}{\exp{\frac{hc}{\lambda k_{B}T}}-1}$$

对于每个测量波长$\lambda_{i}$,我们有理论预测:$$\hat{u_{i}}(T)=u_{\lambda{i}}(T)$$
然后定义均方损失函数，因为连续可微，所以可以自动微分.

但是我们假定了这是绝对黑体，一般地，我们有$$u_{real}(\lambda,T)=\varepsilon(\lambda)u_{\lambda}(T)$$,我们在不知道发射率和不知道温度的时候，我们无法唯一的确定，因为多个未知的参数可能产生相似的观测结果

1.1.4 计算二阶导数可能会遇到什么问题？这些问题可以如何解决？

answer：

假设我们已经求导了一次，如果直接执行x.grad
由于一阶梯度计算后计算图不一定被保留，可能无法继续计算
因此可以加入create_graph = True来为梯度计算过程构建计算图，以便继续求导


In [2]:
import torch
x = torch.tensor(2.0, requires_grad=True)
y = x ** 3

# 计算一阶导数，同时构建梯度的计算图
dy_dx = torch.autograd.grad(
    y, x, create_graph=True
)[0]

# 对一阶导数继续求导
d2y_dx2 = torch.autograd.grad(dy_dx, x)[0]

print(dy_dx)    # tensor(12.)
print(d2y_dx2)  # tensor(12.)

tensor(12., grad_fn=<MulBackward0>)
tensor(12.)


1.1.5 为什么squraed_loss中需要使用reshape?

answer：

def squared_loss(y_hat, y):
    return (y_hat - y.reshape(y_hat.shape)) ** 2 / 2
如果没有reshape
那么y_hat和y的形状可能不同，会触发广播机制

1.1.6 尝试不同的学习率，观察损失函数值下降的快慢

answer：

过小收敛过慢
过快导致震荡

1.1.7 批量大小不能整除样本数量时会发生什么？

answer：

只对最后一组小批量数据产生影响，前面若干组都是批量大小，最后一组是批量大小模样本数量的余数，比如一共10组数据，一个小批量是4组，那么切完就是4，4，2三组